# LFWA: full noise sweep on A100

Standalone follow-up to CelebA using the same five seeds, ten positive anchor-noise levels, ten private-data-noise levels, and four realizations per positive level. The participant count remains p=5. **Select A100 GPU, then Run all**. The notebook verifies the actual GPU and stops if it is not an A100.

This is a fresh 440-fit experiment under `results/lfwa_full_sweep`. The earlier 160-fit experiment remains under `results/lfwa`. No fitted models, deployment projections, or attack scores are imported from the earlier run. Only the hash-verified public source archives and frozen public auditor weights may be copied to avoid downloading them again. All five projections, model fits, and MP/AM/OP evaluations are recomputed. Running this notebook again after interruption resumes only checkpoints belonging to this full-sweep protocol.

## Dataset and source audit

Use the original **40 annotated attributes** of LFWA, including **Smiling**, from the [authors' release](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html). These are not the separate 73 predicted LFW attribute scores. The actual `lfw_att_40.mat` has **13,143 labelled images**, **5,721 identities**, and **1,670 identities with at least two labelled images**. Its paired archive has 13,233 images; the 90 images lacking these annotations are explicitly excluded and listed in the audit. File sizes and SHA256 digests are pinned. Filename joins are validated; unexpected missing, duplicated, or changed images cause an error.

## Participant and utility splits

For each of five seeds:

1. Select **500 eligible identities**, each with at least two labelled images, and assign exactly **100 identities to each of five participants**. One identity belongs to only one participant.
2. Reserve one image per selected identity as its clean identity reference. That reference is excluded from public SVD data, utility training, validation, test inputs and uploaded records.
3. Assign each participant's identities to **70 training / 15 validation / 15 test identities**. All non-reference images of an identity stay in that one split. Thus a two-image identity contributes one reference and one uploaded image in exactly one utility split; it is not spread across all three splits. Assignment balances image volume and Smiling labels subject to those identity quotas. Counts of identities and images are recorded separately, and every utility split must contain both classes.
4. Select **2,000 public images from other identities**, including singletons when selected; public identities are disjoint from every private participant identity. Fit a fresh uncentered h=400 SVD for that seed. This projection is then fixed across all methods and noise draws in the deployment.

Utility is binary **Smiling** prediction. Every image is center-cropped to a square, resized to RGB64 and scaled to [0,1]. Use the same MLP 400→256→128→1, ReLU, Adam lr=0.001, batch 256, maximum 100 epochs and patience 10 as CelebA. Select by validation balanced accuracy and report pooled test balanced accuracy. Local-only fits one model using only each participant's own data, then pools the five held-out confusion matrices. Five participants is the sole count: this notebook does not repeat the CelebA scaling study.

## Identity linkage: no newly trained identity classifier

The privacy evaluator uses a **frozen public VGGFace2 InceptionResnetV1**, not a 100-class classifier trained on LFWA identities. Participant 0 is the colluder/reference; participant 1 is the fixed target.

- Choose one actual uploaded image per target identity (100 queries, spanning the utility splits). Each query's correct candidate is its different, reserved image of the same identity.
- For each query, create ten fixed lineups. Each contains its correct reference and nine distinct decoy identities, sampled from the target's 100-reference pool. Randomize the correct position. Preserve the same lineups for all methods, levels and realizations within the seed.
- Reconstruct the reduced query using each original attack, **MP, AM and OP**, decode through the public SVD basis, and compare its frozen face embedding with the ten clean candidate embeddings by cosine similarity. The highest similarity is the predicted identity. Report the correct-identity fraction over **100 × 10 = 1,000 trials**; random-choice chance is **10%**.
- AM uses the actual honest p=5 alignment and participant 0's disclosed rotation. MP and OP use the target anchor pair. The three attacks share one utility fit and one release. For the common-transform private-noise comparator, invert the known common transform using each query's exact uploaded private noise; this is reported as `known-common-transform`. No reconstructed-image privacy number is assigned to I-GDP or Local-only.

## Full sweep matching CelebA

| Component | Design | Utility fits |
|---|---|---:|
| Noisy anchors | v = 0.006, 0.012, ..., 0.060; 4 draws; 5 seeds | 200 |
| C-GDP private-data noise | σ = 0.3, 0.6, ..., 3.0; 4 draws; 5 seeds | 200 |
| Noiseless C-GDP, I-GDP, AA-GDP | once per method and seed | 15 |
| Local-only | 5 participants × 5 seeds | 25 |
| **Total** | | **440** |

Anchor and private-noise scales are absolute entrywise SDs; these grids are not an equal-noise-budget claim. Zero is represented by the controls. Each realization is an alternative release, not repeated releases given jointly to an attacker. GPM uses float64, spectral initialization, tolerance 1e-7 and a 500-iteration cap; capped runs are retained and flagged. The four draws are averaged within seed/level, then mean and **SD across the five seed means** are reported. Repeated candidate lineups are not treated as independent deployment seeds.

## Outputs and interpretation

Each completed model fit is checkpointed atomically. `raw_results.csv`, `seed_means.csv` and `summary.csv` hold utility and OP linkage; `attack_results.csv`, `attack_seed_means.csv` and `attack_summary.csv` contain the separate attack results. `privacy_utility.pdf` has MP/AM/OP panels. Source manifests, exact participant allocations, reference/query rows, lineups, projection hashes, model histories and convergence diagnostics are saved. The five local aggregate rows are summaries, not additional fits.

The result supports a proof of concept under this reconstruction/auditor evaluation. Images per person and per participant vary substantially in LFWA; the notebook records them and preserves all non-reference labelled images of selected identities. Similarity of the two celebrity-image benchmarks limits the strength of a broad generalization claim.


In [ ]:
# Install only the auditor package; keep Colab's matching torch/torchvision builds.
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import subprocess
import sys
import importlib.metadata
try:
    installed = importlib.metadata.version("facenet-pytorch")
except importlib.metadata.PackageNotFoundError:
    installed = None
if installed != "2.6.0":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--no-deps", "facenet-pytorch==2.6.0"])
import h5py  # Original MATLAB 7.3 attribute annotations.
from google.colab import drive
drive.mount("/content/drive")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Use Runtime > Change runtime type and select a GPU, then Run all.")
gpu_name = torch.cuda.get_device_name()
if "A100" not in gpu_name.upper():
    raise RuntimeError(f"This full LFWA rerun requires an A100 GPU; connected: {gpu_name}. Select Runtime > Change runtime type > A100 GPU.")
print("Verified A100 GPU:", gpu_name)


In [ ]:
CONFIG = {'seeds': [0, 1, 2, 3, 4], 'realizations': 4, 'anchor_scales': [0.006, 0.012, 0.018, 0.024, 0.03, 0.036, 0.042, 0.048, 0.054, 0.06], 'participant_counts': [5], 'private_sigmas': [0.3, 0.6, 0.9, 1.2, 1.5, 1.8, 2.1, 2.4, 2.7, 3.0], 'h': 400, 'r': 1600, 'target_participant': 1, 'reference_participant': 0, 'gpm_tolerance': 1e-07, 'gpm_max_iters': 500, 'mlp': {'hidden_dims': [256, 128], 'max_epochs': 100, 'patience': 10, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 0.0}, 'auditor_cache': '/content/drive/MyDrive/Noisy Anchor Alignment/results/celeba/cache/torch_home', 'output_root': '/content/drive/MyDrive/Noisy Anchor Alignment/results/lfwa_full_sweep', 'protocol': 'lfwa-original40-mp-am-op-full-sweep-a100-v2', 'required_gpu': 'A100', 'source_archive_cache': '/content/drive/MyDrive/Noisy Anchor Alignment/results/lfwa/cache/source', 'gallery_pool': '100 reserved references of target participant, fixed across all conditions', 'noise_units': 'absolute entrywise SD in reduced representation', 'local_baseline': 'one own-data MLP per participant, pool held-out confusion counts'}
CODE_SHA256 = 'c18e01446f4ce2fabfa1df69aa013f2fdb08342267948f6e98c84e2467342781'


## lfwa data
Full standalone implementation.


In [ ]:
"""Original author LFWA 40-attribute release: source audit and fresh deployments."""
import hashlib
import html
from html.parser import HTMLParser
import json
import os
from pathlib import Path, PurePosixPath
import shutil
import tempfile
import urllib.parse
import urllib.request
import zipfile

import h5py
import numpy as np
import pandas as pd
from PIL import Image
import torch

LFWA_SCHEMA = "lfwa-original40-fresh-five-participant-v1"
LFWA_SOURCE_PAGE = "https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html"
LFWA_SOURCE_FOLDER = "https://drive.google.com/drive/folders/0B7EVK8r0v71pQ3NzdzRhVUhSams?resourcekey=0-Kpdd6Vctf-AdJYfS55VULA"
LFWA_FILES = {
    "lfw_att_40.mat": {"id": "0B7EVK8r0v71pZ1ZVTWhrZGNxdzA", "bytes": 6171224,
                       "sha256": "c2724276678a10d6496af5b732592226b719a4ba29eaca41d54c1fe651d2269f"},
    "lfw.zip": {"id": "0B7EVK8r0v71pZDFOOGxhbm1oakE", "bytes": 191778528,
                "sha256": "7cf7e4540012e778a2f0341f0242aca66de3209ac3257acb4fb8a8c7e75cee82"},
}


def _lfwa_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def _lfwa_seed(seed, *parts):
    return int(_lfwa_hash([int(seed), *parts])[:15], 16)


def _lfwa_sha(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


def _lfwa_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, sort_keys=True) + "\n")
    os.replace(tmp, path)


def _lfwa_order(seed, label, values):
    return sorted(values, key=lambda value: _lfwa_hash([int(seed), label, value]))


class _LFWAForm(HTMLParser):
    def __init__(self):
        super().__init__()
        self.action, self.values, self.inside = None, {}, False

    def handle_starttag(self, tag, attrs):
        attrs = dict(attrs)
        if tag == "form" and attrs.get("id") == "download-form":
            self.inside, self.action = True, attrs.get("action")
        if tag == "input" and self.inside and attrs.get("name"):
            self.values[attrs["name"]] = attrs.get("value", "")

    def handle_endtag(self, tag):
        if tag == "form":
            self.inside = False


def _lfwa_download(path, spec):
    """Use the authors' public Drive file and its actual confirmation form."""
    if path.is_file() and path.stat().st_size == spec["bytes"]:
        digest = _lfwa_sha(path)
        if spec.get("sha256", digest) != digest:
            raise RuntimeError(f"Original LFWA source checksum mismatch: {path}")
        return digest
    url = "https://drive.google.com/uc?export=download&id=" + spec["id"]
    response = urllib.request.urlopen(url, timeout=120)
    if "text/html" in response.headers.get("Content-Type", ""):
        parser = _LFWAForm()
        parser.feed(response.read().decode("utf-8"))
        if not parser.action or urllib.parse.urlparse(parser.action).hostname != "drive.usercontent.google.com":
            raise RuntimeError("The authors' LFWA download is unavailable; no replacement data was used")
        response = urllib.request.urlopen(parser.action + "?" + urllib.parse.urlencode(parser.values), timeout=120)
    tmp = path.with_suffix(path.suffix + ".download")
    path.parent.mkdir(parents=True, exist_ok=True)
    with response, tmp.open("wb") as handle:
        shutil.copyfileobj(response, handle)
    digest = _lfwa_sha(tmp)
    if tmp.stat().st_size != spec["bytes"] or spec.get("sha256", digest) != digest:
        raise RuntimeError(f"Downloaded LFWA source does not match the verified release: {path.name}")
    os.replace(tmp, path)
    return digest


def _lfwa_metadata(path):
    with h5py.File(path, "r") as handle:
        def strings(key):
            return ["".join(chr(int(c)) for c in handle[ref][...].ravel())
                    for ref in handle[key][...].ravel()]
        attributes = strings("AttrName")
        names = [name.replace("\\", "/") for name in strings("name")]
        labels = handle["label"][...].T
    if len(attributes) != 40 or attributes.count("Smiling") != 1 or labels.shape != (13143, 40):
        raise RuntimeError("Expected the original LFWA 40-attribute matrix with 13,143 rows")
    if len(names) != len(set(names)) or not np.isin(labels, [0, 1]).all():
        raise RuntimeError("LFWA names must be unique and all 40 annotations binary")
    identities = []
    for name in names:
        parts = PurePosixPath(name).parts
        if len(parts) != 2 or parts[1].rsplit("_", 1)[0] != parts[0]:
            raise RuntimeError(f"Unexpected LFWA identity/filename convention: {name}")
        identities.append(parts[0])
    identity_names = sorted(set(identities))
    if len(identity_names) != 5721:
        raise RuntimeError("Original LFWA annotations must contain 5,721 identities")
    mapping = {name: i for i, name in enumerate(identity_names)}
    frame = pd.DataFrame({"image_id": names, "identity": [mapping[n] for n in identities],
                          "identity_name": identities, "label": labels[:, attributes.index("Smiling")].astype(np.int64)})
    return frame.sort_values("image_id").reset_index(drop=True), attributes


def prepare_lfwa_source(new_cache: Path):
    new_cache = Path(new_cache)
    raw = new_cache / "source"
    raw.mkdir(parents=True, exist_ok=True)
    hashes = {name: _lfwa_download(raw / name, spec) for name, spec in LFWA_FILES.items()}
    frame, attributes = _lfwa_metadata(raw / "lfw_att_40.mat")
    with zipfile.ZipFile(raw / "lfw.zip") as archive:
        members = {}
        for member in archive.namelist():
            if not member.lower().endswith(".jpg") or member.startswith("__MACOSX/"):
                continue
            key = "/".join(PurePosixPath(member.replace("\\", "/")).parts[-2:])
            if key in members:
                raise RuntimeError(f"Duplicate LFWA image path in source archive: {key}")
            members[key] = member
        expected = set(frame.image_id)
        missing, extra = sorted(expected - set(members)), sorted(set(members) - expected)
        # The actual author archive has 13,233 images; its original 40-label MAT
        # annotates 13,143 of them. Record and exclude the known 90 unlabelled
        # images explicitly. Never silently discard a missing annotated image.
        if missing or len(extra) != 90 or len(members) != 13233:
            _lfwa_json(raw / "join_mismatch.json", {"missing_annotated_images": missing, "unannotated_images": extra})
            raise RuntimeError("LFWA image/attribute coverage differs from the verified author release; see source/join_mismatch.json")
        counts = frame.groupby("identity").size()
        if int((counts >= 2).sum()) != 1670:
            raise RuntimeError("Original LFWA must have 1,670 identities with at least two images")
        source_spec = {"schema": LFWA_SCHEMA, "source_page": LFWA_SOURCE_PAGE, "source_folder": LFWA_SOURCE_FOLDER,
                       "files": {name: {**LFWA_FILES[name], "sha256": digest} for name, digest in hashes.items()},
                       "preprocess": "center-square-PIL-bilinear-64x64-RGB-CHW-uint8",
                       "image_selection": "all and only the 13,143 images with original 40-attribute labels",
                       "excluded_unannotated_images_sha256": _lfwa_hash(extra),
                       "image_names_sha256": _lfwa_hash(frame.image_id.tolist())}
        source_hash = _lfwa_hash(source_spec)
        rgb_path, info_path = new_cache / "rgb64.npy", new_cache / "source_manifest.json"
        local = Path(tempfile.gettempdir()) / "naa_lfwa" / source_hash[:20]
        local.mkdir(parents=True, exist_ok=True)
        local_rgb = local / "rgb64.npy"
        if info_path.is_file() and rgb_path.is_file():
            info = json.loads(info_path.read_text())
            if info["source_hash"] != source_hash or _lfwa_sha(rgb_path) != info["rgb_sha256"]:
                raise RuntimeError("LFWA source cache configuration/content mismatch")
            if not local_rgb.is_file() or _lfwa_sha(local_rgb) != info["rgb_sha256"]:
                shutil.copyfile(rgb_path, local_rgb)
        else:
            print("Preparing 13,143 original LFWA images at 64x64 RGB", flush=True)
            pixels = np.lib.format.open_memmap(local_rgb, mode="w+", dtype=np.uint8, shape=(len(frame), 3, 64, 64))
            for i, name in enumerate(frame.image_id):
                with archive.open(members[name]) as data, Image.open(data) as image:
                    image = image.convert("RGB")
                    w, h = image.size
                    side = min(w, h)
                    left, top = (w-side)//2, (h-side)//2
                    image = image.crop((left, top, left+side, top+side)).resize((64, 64), Image.Resampling.BILINEAR)
                    pixels[i] = np.asarray(image, dtype=np.uint8).transpose(2, 0, 1)
            pixels.flush()
            del pixels
            tmp = rgb_path.with_suffix(".tmp.npy")
            shutil.copyfile(local_rgb, tmp)
            digest = _lfwa_sha(local_rgb)
            if _lfwa_sha(tmp) != digest:
                raise RuntimeError("LFWA persistent RGB copy failed content verification")
            os.replace(tmp, rgb_path)
            info = {"source_hash": source_hash, "spec": source_spec, "rgb_sha256": digest,
                    "images": len(frame), "identities": len(counts), "eligible_identities_ge2": int((counts >= 2).sum()),
                    "identity_image_count_histogram": {str(k): int(v) for k, v in counts.value_counts().sort_index().items()},
                    "identity_image_counts": dict(zip(frame.drop_duplicates("identity").identity_name,
                                                       counts.sort_index().astype(int).tolist())),
                    "attribute_names": attributes, "smiling_column_zero_based": attributes.index("Smiling"),
                    "smiling_positive_images": int(frame.label.sum()), "exact_annotated_subset_join": True,
                    "archive_images": len(members), "excluded_unannotated_images": extra,
                    "excluded_unannotated_count": len(extra), "missing_annotated_images": []}
            _lfwa_json(info_path, info)
    pixels = np.load(local_rgb, mmap_mode="r")
    if pixels.shape != (13143, 3, 64, 64) or pixels.dtype != np.uint8:
        raise RuntimeError("LFWA cached RGB shape/dtype mismatch")
    return {"frame": frame, "pixels": pixels, "source_hash": source_hash,
            "source_manifest": info, "new_cache": new_cache}


def _lfwa_allocate(frame, seed, max_participants=5):
    groups = {int(identity): group.index.tolist() for identity, group in frame.groupby("identity")}
    eligible = [identity for identity, rows in groups.items() if len(rows) >= 2]
    private_ids = _lfwa_order(seed, "private-identities", eligible)[:100*max_participants]
    if len(private_ids) != 100*max_participants:
        raise RuntimeError("Not enough LFWA identities with distinct reference/upload images")
    private_set = set(private_ids)
    public_ids, n = [], 0
    for identity in _lfwa_order(seed, "public-identities", [i for i in groups if i not in private_set]):
        public_ids.append(identity)
        n += len(groups[identity])
        if n >= 2000:
            break
    public_rows = _lfwa_order(seed, "public-images", [row for i in public_ids for row in groups[i]])[:2000]
    if len(public_rows) != 2000:
        raise RuntimeError("Insufficient identity-disjoint public images for LFWA SVD")
    stats = []
    for identity in private_ids:
        rows = _lfwa_order(seed, f"identity-{identity}-images", groups[identity])
        stats.append({"identity": identity, "identity_name": str(frame.loc[rows[0], "identity_name"]),
                      "reference_row": rows[0], "rows": rows[1:], "n": len(rows)-1,
                      "positive": int(frame.loc[rows[1:], "label"].sum())})
    caps = {(p, s): n for p in range(max_participants) for s, n in (("train", 70), ("val", 15), ("test", 15))}
    buckets, loads, positive = {k: [] for k in caps}, {k: 0 for k in caps}, {k: 0 for k in caps}
    mean_n = sum(s["n"] for s in stats)/len(stats)
    prevalence = sum(s["positive"] for s in stats)/sum(s["n"] for s in stats)
    stats.sort(key=lambda s: (-s["n"], -abs(s["positive"]/s["n"]-prevalence), _lfwa_hash([seed, s["identity"]])))
    for item in stats:
        def score(key):
            target = mean_n*caps[key]
            def cost(n, pos):
                return (n-target)**2/target + .2*(pos-prevalence*n)**2/max(1, target*prevalence*(1-prevalence))
            delta = cost(loads[key]+item["n"], positive[key]+item["positive"])-cost(loads[key], positive[key])
            return delta+.001*(len(buckets[key])+1)/caps[key], _lfwa_hash([seed, item["identity"], key])
        key = min((k for k in caps if len(buckets[k]) < caps[k]), key=score)
        buckets[key].append(item)
        loads[key] += item["n"]
        positive[key] += item["positive"]
    assignments = [{"participant": p, "split": split, **item} for (p, split), items in sorted(buckets.items())
                   for item in sorted(items, key=lambda a: a["identity"])]
    for key, cap in caps.items():
        if len(buckets[key]) != cap or not 0 < positive[key] < loads[key]:
            raise RuntimeError(f"LFWA split {key} lacks expected identity count or both Smiling classes; no automatic reseeding")
    refs = {a["reference_row"] for a in assignments}
    uploads = {row for a in assignments for row in a["rows"]}
    assert len(private_set) == len(assignments) and not private_set & set(public_ids)
    assert not refs & uploads and not set(public_rows) & (refs | uploads)
    return public_ids, public_rows, assignments


def prepare_lfwa_deployment(source, seed, outdir, h=400, max_participants=5, device=None):
    if max_participants != 5:
        raise ValueError("This standalone LFWA protocol fixes five participants")
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    frame, pixels = source["frame"], source["pixels"]
    public_ids, public_rows, assignments = _lfwa_allocate(frame, seed, max_participants)
    spec = {"schema": LFWA_SCHEMA, "source_hash": source["source_hash"], "seed": int(seed), "h": h,
            "public_images": 2000, "public_identity_ids": public_ids, "public_rows": public_rows,
            "uncentered": True, "svd_q": h+20, "svd_niter": 4, "max_participants": 5,
            "target_participant": 1, "colluder_participant": 0, "assignments": assignments,
            "gallery_pool": "target participant's 100 reserved clean references; ten 10-way lineups per query"}
    config_hash = _lfwa_hash(spec)
    cache = source["new_cache"] / "deployments" / config_hash[:20]
    cache.mkdir(parents=True, exist_ok=True)
    data_path, manifest_path = cache / "deployment.pt", cache / "manifest.json"
    outdir = Path(outdir)
    if data_path.is_file() and manifest_path.is_file():
        manifest = json.loads(manifest_path.read_text())
        if manifest["config_hash"] != config_hash or _lfwa_sha(data_path) != manifest["data_sha256"]:
            raise RuntimeError("LFWA deployment cache configuration/content mismatch")
        result = torch.load(data_path, map_location="cpu", weights_only=True)
        result["manifest"] = manifest
        _lfwa_json(outdir / "deployment_manifest.json", manifest)
        return result
    matrix = torch.from_numpy(np.array(pixels[public_rows])).flatten(1).to(device, torch.float32).div_(255)
    print(f"LFWA seed {seed}: fresh uncentered public SVD on 2,000 images", flush=True)
    with torch.random.fork_rng(devices=[device.index or 0] if device.type == "cuda" else []):
        torch.manual_seed(_lfwa_seed(seed, "public-svd"))
        _, singular, candidate = torch.pca_lowrank(matrix, q=h+20, center=False, niter=4)
    F = torch.linalg.qr(candidate[:, :h].double().cpu(), mode="reduced")[0].float()
    residual = float(torch.linalg.matrix_norm(F.double().T @ F.double()-torch.eye(h)))
    if residual > 1e-3:
        raise RuntimeError("LFWA public projection orthogonality failed")
    del matrix, candidate
    basis = F.to(device)
    def project(rows):
        result = []
        for start in range(0, len(rows), 512):
            batch = torch.from_numpy(np.array(pixels[rows[start:start+512]])).flatten(1).to(device, torch.float32).div_(255)
            result.append((batch @ basis).cpu())
        return torch.cat(result)
    participants, samples, query_q, query_rows, query_ids, query_locations, refs = [], [], [], [], [], [], []
    for p in range(5):
        part = {}
        for split in ("train", "val", "test"):
            items = [a for a in assignments if a["participant"] == p and a["split"] == split]
            rows = [row for a in items for row in a["rows"]]
            q = project(rows)
            part[split+"_q"] = q
            part[split+"_y"] = torch.tensor(frame.loc[rows, "label"].to_numpy(), dtype=torch.float32)
            part[split+"_identities"] = torch.tensor(frame.loc[rows, "identity"].to_numpy())
            samples.append({"participant": p, "split": split, "identities": len(items), "images": len(rows),
                            "smiling": int(frame.loc[rows, "label"].sum())})
            if p == 1:
                lookup = {row: j for j, row in enumerate(rows)}
                for a in items:
                    row = _lfwa_order(seed, f"query-{a['identity']}", a["rows"])[0]
                    query_q.append(q[lookup[row]])
                    query_rows.append(row)
                    query_ids.append(a["identity"])
                    query_locations.append({"split": split, "position": lookup[row]})
                    refs.append(a["reference_row"])
        participants.append(part)
    rng = np.random.default_rng(_lfwa_seed(seed, "fixed-target-gallery"))
    lineups, correct = np.empty((100, 10, 10), dtype=np.int64), np.empty((100, 10), dtype=np.int64)
    for i in range(100):
        for episode in range(10):
            candidates = np.r_[i, rng.choice(np.delete(np.arange(100), i), 9, replace=False)]
            rng.shuffle(candidates)
            lineups[i, episode], correct[i, episode] = candidates, int(np.flatnonzero(candidates == i)[0])
    assert len(query_q) == 100 and not set(query_rows) & set(refs)
    result = {"participants": participants, "F": F, "query_q": torch.stack(query_q),
              "gallery_images": torch.from_numpy(np.array(pixels[refs])).float().div_(255),
              "query_images": torch.from_numpy(np.array(pixels[query_rows])).float().div_(255),
              "lineup_indices": torch.from_numpy(lineups), "correct_positions": torch.from_numpy(correct),
              "query_locations": query_locations, "query_identities": query_ids}
    tmp = data_path.with_suffix(".tmp")
    torch.save(result, tmp)
    os.replace(tmp, data_path)
    manifest = {"config_hash": config_hash, "spec": spec, "sample_counts": samples, "query_rows": query_rows,
                "reference_rows": refs, "query_identities": query_ids, "lineups": lineups.tolist(),
                "correct_positions": correct.tolist(), "projection_orthogonality_residual": residual,
                "projection_sha256": hashlib.sha256(F.numpy().tobytes()).hexdigest(),
                "retained_singular_values": singular[:h].cpu().tolist(), "data_sha256": _lfwa_sha(data_path),
                "data_path": str(data_path), "source_manifest": source["source_manifest"]}
    _lfwa_json(manifest_path, manifest)
    _lfwa_json(outdir / "deployment_manifest.json", manifest)
    result["manifest"] = manifest
    return result


## models protocol
Full standalone implementation.


In [ ]:
"""Standalone CelebA protocol, MLP, and frozen reconstruction-auditor component.

Records are row vectors. Protocol arithmetic is float64; utility and face models
use float32. Rates returned by this module are fractions, not percentages.
"""

import hashlib
import json
import os
import random
import time
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import torch
from torch import nn
from torch.nn import functional as torch_functional


def _model_seed(seed, *parts):
    payload = json.dumps([int(seed), *parts], separators=(",", ":"), ensure_ascii=True)
    return int.from_bytes(hashlib.sha256(payload.encode()).digest()[:4], "little")


def _model_numpy(value, dtype=np.float64):
    if torch.is_tensor(value):
        value = value.detach().cpu().numpy()
    return np.asarray(value, dtype=dtype)


def _model_sync(device):
    if torch.device(device).type == "cuda":
        torch.cuda.synchronize(device)


def _model_haar(rng, h):
    q, r = np.linalg.qr(rng.standard_normal((h, h)))
    signs = np.where(np.diag(r) >= 0.0, 1.0, -1.0)
    return q * signs[None, :]


def generate_bank(seed, pmax=50, h=400, r=1600):
    """Stable participant-indexed draws; changing pmax cannot change a prefix.

    A is centered and satisfies A.T @ A = I. O_i are independent Haar draws
    on O(h); translations have independent N(0,1) coordinates.
    """
    if not (1 <= h < r and pmax >= 1):
        raise ValueError("Require 1 <= h < r and pmax >= 1.")
    rng = np.random.default_rng(_model_seed(seed, "anchor"))
    a = rng.standard_normal((r, h))
    a -= a.mean(axis=0)
    a, triangular = np.linalg.qr(a, mode="reduced")
    a *= np.where(np.diag(triangular) >= 0, 1.0, -1.0)[None, :]
    rotations, translations = [], []
    for participant in range(pmax):
        rotations.append(_model_haar(np.random.default_rng(
            _model_seed(seed, "rotation", participant)), h))
        translations.append(np.random.default_rng(
            _model_seed(seed, "translation", participant)).standard_normal(h))
    return {"A": a, "O": np.stack(rotations), "psi": np.stack(translations),
            "seed": int(seed), "h": int(h), "r": int(r), "pmax": int(pmax)}


def noisy_anchor_bank(bank, seed, level_index, realization, v):
    """One alternative release. Noise is iid over entries and participants.

    A level/realization's participant prefix is independent of requested p.
    This does not center W before release: its mean perturbs the translation.
    """
    if not np.isfinite(v) or v < 0:
        raise ValueError("Anchor scale must be finite and nonnegative.")
    if "clean_anchors" not in bank:
        bank["clean_anchors"] = [bank["A"] @ o + psi
                                 for o, psi in zip(bank["O"], bank["psi"])]
    if v == 0:
        return [b.copy() for b in bank["clean_anchors"]]
    return [b + float(v) * np.random.default_rng(_model_seed(
        seed, "anchor-noise", int(level_index), int(realization), participant
    )).standard_normal(b.shape)
        for participant, b in enumerate(bank["clean_anchors"])]


def upload_blocks(blocks, bank):
    """Create private-data uploads without perturbation, Y_i = Q_i O_i + psi_i."""
    return [_model_numpy(q) @ bank["O"][i] + bank["psi"][i]
            for i, q in enumerate(blocks)]


def _model_polar(matrix):
    u, _, vh = torch.linalg.svd(matrix, full_matrices=False)
    return u @ vh


def align_anchors(B, device="cuda", tol=1e-7, max_iters=500, reference=0):
    """Ling spectral initialization and unregularized generalized power method.

    No (p*h)-square Gram is allocated. If r < p*h, spectral initialization
    diagonalizes D D.T and recovers right singular vectors as D.T U / s.
    Iteration is exactly R_next = block_polar(D.T D R), as in the old code.
    Stopping uses the OLD absolute Frobenius step <= tol (not relative step).
    The common output gauge is normalized to the reference participant.

    Returns (maps, diagnostics), with maps['rotations_R'] of shape (p,h,h)
    and maps['means_b'] of shape (p,h). Aligned data are
    (Y_i-b_i) @ maps['rotations_R'][i] + b_reference.
    """
    dev = torch.device(device)
    if not B or max_iters < 1 or tol <= 0 or not 0 <= reference < len(B):
        raise ValueError("Invalid anchors, reference, or solver settings.")
    _model_sync(dev)
    started = time.perf_counter()
    blocks = [torch.as_tensor(b, dtype=torch.float64, device=dev) for b in B]
    r, h = blocks[0].shape
    p = len(blocks)
    if any(tuple(b.shape) != (r, h) for b in blocks) or r <= h:
        raise ValueError("Every anchor block must have common shape r x h, with r > h.")
    means = torch.stack([b.mean(dim=0) for b in blocks])
    D = torch.cat([b - b.mean(dim=0) for b in blocks], dim=1)
    del blocks
    if r < p * h:
        left_gram = D @ D.T
        eigenvalues, eigenvectors = torch.linalg.eigh(left_gram)
        eigenvalues = eigenvalues.flip(0)
        top = eigenvalues[:h]
        if float(top[-1].item()) <= torch.finfo(torch.float64).eps * float(top[0].item()):
            raise RuntimeError("Spectral initialization has a numerically deficient top-h subspace.")
        right = (D.T @ eigenvectors[:, -h:].flip(1)) / top.sqrt().unsqueeze(0)
        gap = float((eigenvalues[h - 1] - eigenvalues[h]).item())
        scale = float(top[-1].item())
        del eigenvectors, eigenvalues, left_gram
    else:
        _, singular, vh = torch.linalg.svd(D, full_matrices=False)
        right = vh[:h].T
        scale = float(singular[h - 1].square().item())
        next_value = float(singular[h].square().item()) if len(singular) > h else 0.0
        gap = scale - next_value
        del vh, singular
    stacked = _model_polar(right.reshape(p, h, h)).reshape(p * h, h)
    del right
    _model_sync(dev)
    initialization_seconds = time.perf_counter() - started
    objective_history = [float((D @ stacked).square().sum().item())]
    step_history = []
    converged = False
    iteration_started = time.perf_counter()
    for _ in range(max_iters):
        next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
        step = float(torch.linalg.vector_norm(next_stacked - stacked).item())
        step_history.append(step)
        stacked = next_stacked
        objective_history.append(float((D @ stacked).square().sum().item()))
        if step <= tol:
            converged = True
            break
    _model_sync(dev)
    iteration_seconds = time.perf_counter() - iteration_started
    next_stacked = _model_polar((D.T @ (D @ stacked)).reshape(p, h, h)).reshape(p*h, h)
    fixed_absolute = float(torch.linalg.vector_norm(next_stacked - stacked).item())
    fixed_relative = fixed_absolute / max(1.0, float(torch.linalg.vector_norm(stacked).item()))
    rotations = stacked.reshape(p, h, h)
    # This exact gauge normalization is also applied to the aligned mean.
    rotations = rotations @ rotations[reference].T.clone()
    rotations[reference] = torch.eye(h, device=dev, dtype=torch.float64)
    template = sum((D[:, i*h:(i+1)*h] @ rotations[i] for i in range(p))) / p
    residual = sum(float((D[:, i*h:(i+1)*h] @ rotations[i] - template).square().sum().item())
                   for i in range(p))
    denominator = float(D.square().sum().item())
    maps = {"rotations_R": rotations.detach().cpu().numpy(),
            "means_b": means.detach().cpu().numpy(), "reference": int(reference)}
    _model_sync(dev)
    diagnostics = {
        "algorithm": "spectral-unregularized-GPM",
        "dtype": "float64", "max_iters": int(max_iters), "tolerance": float(tol),
        "stopping_rule": "absolute Frobenius norm of successive stacked iterates",
        "iterations": len(step_history), "converged": bool(converged),
        "termination_reason": "step_tolerance" if converged else "iteration_cap",
        "last_step": step_history[-1], "fixed_point_residual": fixed_relative,
        "fixed_point_residual_absolute": fixed_absolute,
        "gopp_residual": residual, "relative_gopp_residual": residual / max(denominator, 1e-300),
        "eigengap": gap, "relative_eigengap": gap / max(scale, 1e-300),
        "objective_history": objective_history, "step_history": step_history,
        "initialization_seconds": initialization_seconds, "iteration_seconds": iteration_seconds,
        "elapsed_seconds": time.perf_counter() - started,
        "reference_participant": int(reference), "participant_count": int(p),
    }
    return maps, diagnostics


def apply_maps(uploads, maps):
    means, rotations = maps["means_b"], maps["rotations_R"]
    reference = maps.get("reference", 0)
    return [(_model_numpy(y) - means[i]) @ rotations[i] + means[reference]
            for i, y in enumerate(uploads)]


def op_reconstruct(A, B, Y, device="cuda"):
    """Known-anchor OP recovery of projected records; no image decoder here."""
    dev = torch.device(device)
    a = torch.as_tensor(A, device=dev, dtype=torch.float64)
    b = torch.as_tensor(B, device=dev, dtype=torch.float64)
    y = torch.as_tensor(Y, device=dev, dtype=torch.float64)
    am, bm = a.mean(0), b.mean(0)
    rotation = _model_polar((a - am).T @ (b - bm))
    translation = bm - am @ rotation
    return ((y - translation) @ rotation.T).cpu().numpy()


def _model_set_seed(seed):
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(int(seed))
    np.random.seed(int(seed) % (2**32))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def _model_metrics(logits, labels):
    prediction = np.asarray(logits).reshape(-1) >= 0.0
    truth = np.asarray(labels).reshape(-1).astype(bool)
    counts = [int((truth == label).sum()) for label in (False, True)]
    if not all(counts):
        raise ValueError("Balanced accuracy requires both classes in the evaluation split.")
    recall = [float((prediction[truth == label] == label).mean()) for label in (False, True)]
    return {"balanced_accuracy": float(np.mean(recall)),
            "accuracy": float((prediction == truth).mean()),
            "recall_not_smiling": recall[0], "recall_smiling": recall[1],
            "n_records": int(len(truth))}


def fit_mlp(trainX, trainY, valX, valY, testX, testY, seed, cfg=None):
    """Fit the fixed vector MLP and select the epoch using validation BA only.

    seed must be held fixed within a deployment for paired protocol/noise runs.
    A fresh local permutation generator is reset for each call. No feature-wise
    normalization or test-dependent threshold selection is performed.
    """
    config = {"device": "cuda" if torch.cuda.is_available() else "cpu",
              "max_epochs": 50, "patience": 8, "batch_size": 256,
              "lr": 1e-3, "weight_decay": 0.0, "hidden_dims": (256, 128)}
    if cfg is not None:
        config.update(cfg if isinstance(cfg, dict) else vars(cfg))
    if int(config["max_epochs"]) < 1 or int(config["patience"]) < 1:
        raise ValueError("max_epochs and patience must be positive.")
    _model_set_seed(seed)
    dev = torch.device(config["device"])
    arrays = [_model_numpy(x, np.float32) for x in (trainX, valX, testX)]
    labels = [_model_numpy(y, np.float32).reshape(-1) for y in (trainY, valY, testY)]
    h = arrays[0].shape[1]
    if any(x.ndim != 2 or x.shape[1] != h or len(x) != len(y)
           or not np.isfinite(x).all() or not np.isin(y, [0.0, 1.0]).all()
           for x, y in zip(arrays, labels)):
        raise ValueError("MLP requires finite n x h inputs and matching binary 0/1 labels.")
    for y in labels:
        if len(np.unique(y)) != 2:
            raise ValueError("Each utility split must contain both smiling classes.")
    hidden = tuple(config["hidden_dims"])
    if len(hidden) != 2:
        raise ValueError("Expected exactly two MLP hidden layers.")
    network = nn.Sequential(nn.Linear(h, hidden[0]), nn.ReLU(),
                            nn.Linear(hidden[0], hidden[1]), nn.ReLU(),
                            nn.Linear(hidden[1], 1)).to(dev)
    optimizer = torch.optim.Adam(network.parameters(), lr=float(config["lr"]),
                                 weight_decay=float(config["weight_decay"]))
    criterion = nn.BCEWithLogitsLoss()
    xtrain = torch.from_numpy(np.ascontiguousarray(arrays[0]))
    ytrain = torch.from_numpy(labels[0].copy())
    permutation_generator = torch.Generator(device="cpu").manual_seed(int(seed))
    batch_size = int(config["batch_size"])

    def predict(x):
        network.eval()
        output = []
        with torch.no_grad():
            for start in range(0, len(x), batch_size):
                batch = torch.as_tensor(x[start:start+batch_size], device=dev, dtype=torch.float32)
                output.append(network(batch).reshape(-1).cpu().numpy())
        return np.concatenate(output)

    _model_sync(dev)
    started = time.perf_counter()
    best_score, best_state, best_epoch, stale = -np.inf, None, 0, 0
    history = []
    for epoch in range(1, int(config["max_epochs"]) + 1):
        network.train()
        permutation = torch.randperm(len(xtrain), generator=permutation_generator)
        loss_sum = 0.0
        for start in range(0, len(xtrain), batch_size):
            indices = permutation[start:start+batch_size]
            xb, yb = xtrain[indices].to(dev), ytrain[indices].to(dev)
            optimizer.zero_grad(set_to_none=True)
            logits = network(xb).reshape(-1)
            loss = criterion(logits, yb)
            if not torch.isfinite(loss):
                raise RuntimeError("Nonfinite MLP training loss.")
            loss.backward()
            optimizer.step()
            loss_sum += float(loss.detach().item()) * len(indices)
        val_metrics = _model_metrics(predict(arrays[1]), labels[1])
        score = val_metrics["balanced_accuracy"]
        history.append({"epoch": epoch, "train_loss": loss_sum / len(xtrain),
                        "validation_balanced_accuracy": score})
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {name: value.detach().cpu().clone()
                          for name, value in network.state_dict().items()}
        else:
            stale += 1
        if stale >= int(config["patience"]):
            break
    network.load_state_dict(best_state)
    # Test records are evaluated exactly once, after the validation selection.
    result = _model_metrics(predict(arrays[2]), labels[2])
    _model_sync(dev)
    result.update({"val_best_balanced_accuracy": float(best_score),
                   "best_epoch": int(best_epoch), "epochs": int(epoch),
                   "train_seconds": time.perf_counter() - started,
                   "training_seed": int(seed), "decision_threshold_logit": 0.0,
                   "architecture": [h, *hidden, 1], "training_history": history,
                   "stopped_early": bool(epoch < int(config["max_epochs"]))})
    del network, optimizer, best_state
    return result


class FaceAuditor:
    """Existing frozen VGGFace2 InceptionResnetV1 cross-image identity auditor."""

    def __init__(self, device="cuda", torchcache=None, batch_size=64):
        if torchcache is not None:
            Path(torchcache).mkdir(parents=True, exist_ok=True)
            os.environ["TORCH_HOME"] = str(torchcache)
        from facenet_pytorch import InceptionResnetV1
        self.device = torch.device(device)
        self.batch_size = int(batch_size)
        self.model = InceptionResnetV1(pretrained="vggface2").eval().to(self.device)
        for parameter in self.model.parameters():
            parameter.requires_grad_(False)
        digest = hashlib.sha256()
        for name, value in sorted(self.model.state_dict().items()):
            digest.update(name.encode())
            digest.update(value.detach().cpu().contiguous().numpy().tobytes())
        self.metadata = {
            "architecture": "InceptionResnetV1", "pretrained": "vggface2",
            "frozen": True, "state_sha256": digest.hexdigest(),
            "preprocessing": "RGB [0,1], bilinear 160x160, (255*x-127.5)/128",
            "comparison": "cosine similarity to clean cross-image references",
        }

    @torch.no_grad()
    def encode_gallery(self, images):
        """Accept uint8 or [0,1] float RGB, NCHW or NHWC, return n x 512 numpy."""
        tensor = torch.as_tensor(images)
        if tensor.ndim != 4:
            raise ValueError("Expected a four-dimensional RGB image batch.")
        if tensor.shape[1] != 3 and tensor.shape[-1] == 3:
            tensor = tensor.permute(0, 3, 1, 2)
        if tensor.shape[1] != 3:
            raise ValueError("Auditor images must have three RGB channels.")
        uint8 = tensor.dtype == torch.uint8
        embeddings = []
        for start in range(0, len(tensor), self.batch_size):
            batch = tensor[start:start+self.batch_size].to(self.device, dtype=torch.float32)
            if uint8:
                batch = batch / 255.0
            if not torch.isfinite(batch).all() or float(batch.min()) < -1e-6 or float(batch.max()) > 1.000001:
                raise ValueError("Auditor floating images must be finite and in [0,1].")
            batch = torch_functional.interpolate(batch, size=(160, 160), mode="bilinear", align_corners=False)
            batch = (batch * 255.0 - 127.5) / 128.0
            embeddings.append(torch_functional.normalize(self.model(batch), dim=1).cpu())
        if not embeddings:
            return np.empty((0, 512), dtype=np.float32)
        return torch.cat(embeddings).numpy()

    @torch.no_grad()
    def linkage(self, qhat, F, gallery_embeddings, lineup_indices, correct_pos):
        """Decode an orthonormal public-SVD representation and score fixed lineups.

        qhat: n_query x h; F: (3*64*64) x h, using CHW flattening.
        lineups: n_query x n_repeated_galleries x n_candidates (or n_query x k).
        Returns identity-balanced linkage when each query represents one identity.
        """
        query = _model_numpy(qhat, np.float32)
        basis = torch.as_tensor(F, dtype=torch.float32, device=self.device)
        if basis.shape[0] != 3 * 64 * 64 or basis.shape[1] != query.shape[1]:
            raise ValueError("F and qhat must use the public RGB64 SVD projection.")
        query_embeddings = []
        for start in range(0, len(query), self.batch_size):
            reduced = torch.as_tensor(query[start:start+self.batch_size], device=self.device)
            decoded = (reduced @ basis.T).clamp(0.0, 1.0).reshape(-1, 3, 64, 64)
            query_embeddings.append(self.encode_gallery(decoded))
        query_embeddings = np.concatenate(query_embeddings)
        gallery = _model_numpy(gallery_embeddings, np.float32)
        indices = np.asarray(lineup_indices, dtype=np.int64)
        truth = np.asarray(correct_pos, dtype=np.int64)
        if indices.ndim == 2:
            indices, truth = indices[:, None, :], truth.reshape(-1, 1)
        if indices.ndim != 3 or len(indices) != len(query) or truth.shape != indices.shape[:2]:
            raise ValueError("Lineup indices and correct positions do not match query count.")
        if (indices < 0).any() or (indices >= len(gallery)).any():
            raise ValueError("Gallery index outside available reference embeddings.")
        if (truth < 0).any() or (truth >= indices.shape[-1]).any():
            raise ValueError("Correct position outside candidate lineup.")
        scores = np.einsum("qd,qgkd->qgk", query_embeddings, gallery[indices])
        success = scores.argmax(axis=-1) == truth
        return {"linkage_rate": float(success.mean()),
                "per_query_success": success.mean(axis=1).tolist(),
                "n_queries": int(len(query)), "lineups_per_query": int(indices.shape[1]),
                "n_candidates": int(indices.shape[2]), "n_lineups": int(success.size),
                "chance_rate": 1.0 / indices.shape[2]}


def protocol_self_test():
    """Small CPU algebra and reference-gauge checks, without pretrained weights."""
    bank = generate_bank(173, pmax=3, h=5, r=17)
    short = generate_bank(173, pmax=2, h=5, r=17)
    np.testing.assert_array_equal(bank["O"][:2], short["O"])
    np.testing.assert_array_equal(bank["A"], short["A"])
    np.testing.assert_allclose(bank["A"].mean(0), 0, atol=2e-15)
    np.testing.assert_allclose(bank["A"].T @ bank["A"], np.eye(5), atol=2e-14)
    data = [np.random.default_rng(i).normal(size=(11, 5)) for i in range(3)]
    uploads = upload_blocks(data, bank)
    clean = noisy_anchor_bank(bank, 173, 0, 0, 0.0)
    maps, diag = align_anchors(clean, device="cpu", tol=1e-10, max_iters=100)
    aligned = apply_maps(uploads, maps)
    for i, result in enumerate(aligned):
        np.testing.assert_allclose(result, data[i] @ bank["O"][0] + bank["psi"][0], atol=1e-11)
        recovered = op_reconstruct(bank["A"], clean[i], uploads[i], device="cpu")
        np.testing.assert_allclose(recovered, data[i], atol=1e-11)
    # Coupling isotropic private noise by orthogonal rotation yields the exact
    # common-reference noisy matrix asserted by the equivalence result.
    errors = [0.1 * np.random.default_rng(500+i).normal(size=q.shape)
              for i, q in enumerate(data)]
    noisy_uploads = [y + errors[i] @ bank["O"][i] for i, y in enumerate(uploads)]
    noisy_aligned = apply_maps(noisy_uploads, maps)
    for i, result in enumerate(noisy_aligned):
        common = (data[i] + errors[i]) @ bank["O"][0] + bank["psi"][0]
        np.testing.assert_allclose(result, common, atol=1e-11)
    np.testing.assert_array_equal(maps["rotations_R"][0], np.eye(5))
    # Common right multiplication must disappear in reference normalization.
    gauge = _model_haar(np.random.default_rng(77), 5)
    raw = maps["rotations_R"] @ gauge
    normalized = raw @ raw[0].T
    np.testing.assert_allclose(normalized, maps["rotations_R"], atol=1e-13)
    noise1 = noisy_anchor_bank(bank, 173, 3, 2, 0.024)
    noise2 = noisy_anchor_bank(short, 173, 3, 2, 0.024)
    for a, b in zip(noise1, noise2):
        np.testing.assert_array_equal(a, b)
    # Exercise the r-square spectral route too (p*h > r).
    wide = generate_bank(31, pmax=5, h=5, r=17)
    _, wide_diag = align_anchors(noisy_anchor_bank(wide, 31, 0, 0, 0), device="cpu",
                                tol=1e-10, max_iters=100)
    assert diag["converged"] and wide_diag["converged"]
    return {"passed": True, "noiseless_gopp_relative_residual": diag["relative_gopp_residual"],
            "wide_spectral_relative_residual": wide_diag["relative_gopp_residual"]}


## attack helpers
Full standalone implementation.


In [ ]:
"""The manuscript's three existing known-anchor reconstruction attacks."""


def attack_reconstruct(A, B, Y, method, device="cuda", maps=None, target=1,
                       colluder_rotation=None):
    """Return reconstructed projected records as float64 NumPy rows.

    B is the target participant's r x h anchor block. AM takes the analyst's
    maps plus participant 0's disclosed orthogonal transform. These formulas
    reproduce the original OP, MP, and alignment-calibration attacks.
    """
    method = method.upper()
    if method == "OP":
        return op_reconstruct(A, B, Y, device=device)
    dev = torch.device(device)
    a = torch.as_tensor(A, dtype=torch.float64, device=dev)
    b = torch.as_tensor(B, dtype=torch.float64, device=dev)
    y = torch.as_tensor(Y, dtype=torch.float64, device=dev)
    amean, bmean = a.mean(0), b.mean(0)
    if method == "MP":
        transform = torch.linalg.pinv(a - amean) @ (b - bmean)
        translation = bmean - amean @ transform
        return ((y - translation) @ torch.linalg.pinv(transform)).cpu().numpy()
    if method != "AM":
        raise ValueError("method must be OP, MP, or AM")
    if maps is None or colluder_rotation is None:
        raise ValueError("AM requires analyst alignment maps and the colluder's rotation")
    rotations = torch.as_tensor(maps["rotations_R"], dtype=torch.float64, device=dev)
    colluder = torch.as_tensor(colluder_rotation, dtype=torch.float64, device=dev)
    common = _model_polar(colluder @ rotations[0])
    transform = common @ rotations[target].T
    translation = bmean - amean @ transform
    return ((y - translation) @ transform.T).cpu().numpy()


def attack_self_test():
    bank = generate_bank(501, pmax=3, h=4, r=13)
    anchors = noisy_anchor_bank(bank, 501, 0, 0, 0)
    q = np.random.default_rng(19).normal(size=(9, 4))
    y = q @ bank["O"][1] + bank["psi"][1]
    maps, _ = align_anchors(anchors, device="cpu", tol=1e-10, max_iters=100)
    errors = {}
    for method in ("OP", "MP", "AM"):
        recovered = attack_reconstruct(bank["A"], anchors[1], y, method, device="cpu",
                                       maps=maps, target=1, colluder_rotation=bank["O"][0])
        errors[method] = float(np.linalg.norm(recovered-q) / np.linalg.norm(q))
        np.testing.assert_allclose(recovered, q, atol=1e-11)
    return {"passed": True, "noiseless_relative_errors": errors}


## lfwa runner
Full standalone implementation.


In [ ]:
"""Resumable schedule and seed-level reporting for the standalone notebook."""
import csv
import gc
import importlib.metadata
import platform
import traceback
from datetime import datetime, timezone

SPLITS = ("train", "val", "test")


def _run_completed_counts(root):
    """Count actual fitted models separately from reused local aggregate rows."""
    root = Path(root)
    record_paths = list((root / "records").glob("*.json"))
    aggregate_count = sum("_Local-only_" in path.stem for path in record_paths)
    local_count = sum(1 for _ in (root / "local_models").glob("*.json"))
    return {"completed_fits": len(record_paths) - aggregate_count + local_count,
            "completed_result_rows": len(record_paths),
            "completed_local_fits": local_count,
            "completed_local_aggregates": aggregate_count}


def _run_local_counts(metrics, labels):
    """Recover integer confusion counts from the exact per-class recalls."""
    truth = np.asarray(labels).reshape(-1)
    negative, positive = int((truth == 0).sum()), int((truth == 1).sum())
    if negative + positive != len(truth) or not negative or not positive:
        raise ValueError("Local test labels must contain both binary classes.")
    tn = int(round(metrics["recall_not_smiling"] * negative))
    tp = int(round(metrics["recall_smiling"] * positive))
    if not (0 <= tn <= negative and 0 <= tp <= positive):
        raise RuntimeError("Local confusion counts are inconsistent with test labels.")
    return {"true_negatives": tn, "false_positives": negative - tn,
            "false_negatives": positive - tp, "true_positives": tp}


def _run_pool_local(local_rows):
    """Evaluate all local models' test predictions as one pooled cohort."""
    names = ("true_negatives", "false_positives", "false_negatives", "true_positives")
    counts = {name: sum(row["confusion_counts"][name] for row in local_rows) for name in names}
    tn, fp, fn, tp = (counts[name] for name in names)
    negative, positive = tn + fp, tp + fn
    if negative == 0 or positive == 0:
        raise RuntimeError("Pooled local evaluation requires both classes.")
    return {"balanced_accuracy": 0.5 * (tn / negative + tp / positive),
            "accuracy": (tn + tp) / (negative + positive),
            "recall_not_smiling": tn / negative, "recall_smiling": tp / positive,
            "n_records": negative + positive, "n_local_models": len(local_rows),
            "confusion_counts": counts,
            "aggregation": "pool confusion counts across the participating local models"}


def _run_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n")
    os.replace(temporary, path)


def _run_matmul(x, rotation, shift, device):
    answer = []
    r = torch.as_tensor(rotation, device=device, dtype=torch.float64)
    b = torch.as_tensor(shift, device=device, dtype=torch.float64)
    for start in range(0, len(x), 4096):
        t = torch.as_tensor(x[start:start+4096], device=device, dtype=torch.float64)
        answer.append((t @ r + b).cpu().numpy())
    return np.concatenate(answer)


def _run_collaboration(uploads, maps, p, device):
    result = {}
    for split in SPLITS:
        if maps is None:
            result[split] = np.concatenate(uploads[split][:p]).astype(np.float32)
        else:
            rotations, means = maps["rotations_R"], maps["means_b"]
            reference = maps["reference"]
            result[split] = np.concatenate([
                _run_matmul(uploads[split][i], rotations[i],
                            means[reference] - means[i] @ rotations[i], device).astype(np.float32)
                for i in range(p)])
    return result


def _run_clean_maps(anchors, device):
    b = torch.as_tensor(np.stack(anchors), dtype=torch.float64, device=device)
    means = b.mean(1)
    centered = b - means[:, None, :]
    rotations = _model_polar(centered.transpose(1, 2) @ centered[0])
    rotations[0] = torch.eye(rotations.shape[-1], device=device, dtype=torch.float64)
    return {"rotations_R": rotations.cpu().numpy(), "means_b": means.cpu().numpy(), "reference": 0}


def summarize_results(root, config):
    """First average the four alternative releases, then summarize five deployments."""
    root = Path(root)
    rows = [json.loads(path.read_text()) for path in sorted((root / "records").glob("*.json"))]
    if not rows:
        return pd.DataFrame()
    flat = []
    for row in rows:
        flat.append({key: value for key, value in row.items()
                     if value is None or isinstance(value, (str, int, float, bool))})
    raw = pd.DataFrame(flat)
    raw.to_csv(root / "raw_results.csv", index=False)
    attack_rows = []
    for row in rows:
        for attack, result in (row.get("privacy") or {}).get("attacks", {}).items():
            attack_rows.append({**{key: row[key] for key in ("run_id", "method", "participant_count", "noise_scale", "outer_seed", "realization")},
                                "attack": attack, "linkage_rate": result["linkage_rate"],
                                "projected_recovery_relative_error": result["projected_recovery_relative_error"]})
    if attack_rows:
        attack_raw = pd.DataFrame(attack_rows)
        attack_raw.to_csv(root / "attack_results.csv", index=False)
        akeys = ["method", "participant_count", "noise_scale", "attack", "outer_seed"]
        ametrics = ["linkage_rate", "projected_recovery_relative_error"]
        means = attack_raw.groupby(akeys)[ametrics].mean().reset_index()
        reps = attack_raw.groupby(akeys).size().rename("completed_realizations").reset_index()
        means = means.merge(reps, on=akeys)
        means["seed_cell_complete"] = means.completed_realizations == np.where(means.noise_scale > 0, config["realizations"], 1)
        means.to_csv(root / "attack_seed_means.csv", index=False)
        complete_attacks = means[means.seed_cell_complete]
        combined = complete_attacks.groupby(akeys[:-1])[ametrics].agg(["mean", "std", "count"])
        combined.columns = ["_".join(c) for c in combined.columns]
        combined = combined.reset_index()
        combined["final"] = combined.linkage_rate_count == len(config["seeds"])
        combined.to_csv(root / "attack_summary.csv", index=False)
    keys = ["method", "participant_count", "noise_scale", "outer_seed"]
    metrics = ["balanced_accuracy", "linkage_rate", "fit_seconds", "alignment_seconds"]
    seed_means = raw.groupby(keys, dropna=False)[metrics].mean().reset_index()
    repetitions = raw.groupby(keys).size().rename("completed_realizations").reset_index()
    seed_means = seed_means.merge(repetitions, on=keys)
    seed_means["required_realizations"] = np.where(seed_means.noise_scale > 0, config["realizations"], 1)
    seed_means["seed_cell_complete"] = seed_means.completed_realizations == seed_means.required_realizations
    seed_means.to_csv(root / "seed_means.csv", index=False)
    complete = seed_means[seed_means.seed_cell_complete]
    summary_rows = []
    for key, group in complete.groupby(keys[:3], dropna=False):
        record = dict(zip(keys[:3], key))
        record.update(n_seeds=len(group), final=len(group) == len(config["seeds"]))
        for metric in metrics:
            values = group[metric].dropna()
            record[metric + "_mean"] = values.mean() if len(values) else np.nan
            record[metric + "_seed_sd"] = values.std(ddof=1) if len(values) > 1 else np.nan
        summary_rows.append(record)
    summary = pd.DataFrame(summary_rows)
    summary.to_csv(root / "summary.csv", index=False)
    raw[raw.participant_count == 5].to_csv(root / "main_p5_results.csv", index=False)
    diagnostics = raw[raw.method == "NAA-GDP"]
    cols = [c for c in ("run_id", "outer_seed", "participant_count", "noise_scale", "realization",
                        "gpm_converged", "gpm_iterations", "gpm_last_step", "alignment_seconds") if c in raw]
    diagnostics[cols].to_csv(root / "alignment_diagnostics.csv", index=False)
    counts = _run_completed_counts(root)
    schedule_complete = (len(raw) == expected_result_rows(config)
                         and counts["completed_fits"] == expected_fits(config))
    if len(summary):
        _run_plots(root, summary, schedule_complete)
    noisy = raw[raw.method == "NAA-GDP"].groupby("participant_count").size().to_dict()
    report = ["# LFWA run status", "", f"Completed fitted models: {counts['completed_fits']}/{expected_fits(config)}.",
              f"Completed main result rows: {len(raw)}/{expected_result_rows(config)}.",
              f"Completed individual local fits: {counts['completed_local_fits']}/{len(config['seeds']) * max(config['participant_counts'])}.",
              f"Completed positive-noise NAA fits by participant count: {noisy}.", "",
              "All rates in CSV files are fractions. SD is across outer-seed means, after averaging",
              "four realizations within each seed and noise level. A summary row is final only when",
              "all five seeds have all required realizations. Partial figures are labelled provisional.", "",
              "The main p=5 CSV is an exact filter of raw_results.csv. No rerun or new deployment",
              "is used for the participant-count figure. The same target participant and fixed",
              "100-reference gallery pool are retained across participant counts.", "",
              "Local-only fits one MLP per participant on its own unperturbed public-SVD records.",
              "Each local model is fitted once per seed and reused in nested collaborations.",
              "Local-only result rows pool confusion counts across the first p local models, so",
              "their balanced accuracy uses the same pooled test cohort as collaboration methods.",
              "They are aggregate rows, not additional fits. Their fit_seconds is the sum of the",
              "constituent local fitting times. Privacy is unmeasured, not zero.", "",
              "This evaluates MP/AM/OP reconstructions followed by a frozen public face auditor; it does",
              "not establish privacy against all admissible attacks. I-GDP and Local-only have no linkage estimate."]
    (root / "REPORT.md").write_text("\n".join(report) + "\n")
    return raw


def _run_plots(root, summary, schedule_complete=False):
    import matplotlib.pyplot as plt
    attack_path = Path(root) / "attack_summary.csv"
    if not attack_path.exists():
        return
    attacks = pd.read_csv(attack_path)
    keys = ["method", "participant_count", "noise_scale"]
    paired = attacks.merge(summary[keys + ["balanced_accuracy_mean", "balanced_accuracy_seed_sd"]], on=keys, validate="many_to_one")
    fig, axes = plt.subplots(1, 3, figsize=(12.6, 4), sharey=True)
    for attack, ax in zip(("MP", "AM", "OP"), axes):
        selected = paired[(paired.attack == attack) | (paired.attack == "known-common-transform")]
        for method in ("NAA-GDP", "C-GDP-private-noise", "AA-GDP"):
            series = selected[selected.method == method].sort_values("noise_scale")
            if len(series):
                ax.errorbar(100*series.linkage_rate_mean, 100*series.balanced_accuracy_mean,
                    xerr=100*series.linkage_rate_std.fillna(0), yerr=100*series.balanced_accuracy_seed_sd.fillna(0),
                    marker="o", capsize=2, label=method)
        for method, style in (("I-GDP", "-."), ("Local-only", "--")):
            baseline = summary[summary.method == method]
            if len(baseline):
                ax.axhline(100*baseline.balanced_accuracy_mean.iloc[0], linestyle=style, label=method+" utility")
        ax.axvline(10, color="grey", linestyle=":")
        ax.set(title=attack+" reconstruction", xlabel="Frozen-auditor linkage (%)")
    axes[0].set_ylabel("Smiling balanced accuracy (%)")
    axes[-1].legend(fontsize=7)
    fig.suptitle("LFWA, p=5" + (" (provisional)" if not schedule_complete else ""))
    fig.tight_layout()
    fig.savefig(Path(root)/"privacy_utility.pdf")
    plt.close(fig)


def expected_fits(config):
    s, r, p = len(config["seeds"]), config["realizations"], len(config["participant_counts"])
    return s * (p * (len(config["anchor_scales"]) * r + 3)
                + len(config["private_sigmas"]) * r + max(config["participant_counts"]))


def expected_result_rows(config):
    s, r, p = len(config["seeds"]), config["realizations"], len(config["participant_counts"])
    return s * (p * (len(config["anchor_scales"]) * r + 4) + len(config["private_sigmas"]) * r)


def _run_require_gpu(config):
    if not torch.cuda.is_available():
        raise RuntimeError("Select a Colab A100 GPU runtime before running this experiment.")
    gpu = torch.cuda.get_device_name()
    required = config.get("required_gpu")
    if required and required.upper() not in gpu.upper():
        raise RuntimeError(f"This experiment requires {required}; connected GPU is {gpu}. Change the Colab runtime type.")
    return gpu


def run_experiment(config, code_hash):
    gpu = _run_require_gpu(config)
    root = Path(config["output_root"])
    root.mkdir(parents=True, exist_ok=True)
    identity = {"config": config, "code_sha256": code_hash}
    signature = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    lock = root / "protocol.json"
    if lock.exists() and json.loads(lock.read_text())["signature"] != signature:
        raise RuntimeError("Existing results use a different code/configuration. Do not mix protocols; archive deliberately before changing this experiment.")
    _run_json(lock, {**identity, "signature": signature, "expected_fits": expected_fits(config),
                     "expected_result_rows": expected_result_rows(config)})
    device = "cuda"
    environment = {"python": platform.python_version(), "platform": platform.platform(),
                   "torch": torch.__version__, "cuda": torch.version.cuda,
                   "gpu": gpu, "packages": {
                       name: importlib.metadata.version(name) for name in
                       ("numpy", "pandas", "torch", "torchvision", "facenet-pytorch", "Pillow", "matplotlib")}}
    env_path = root / "environment.json"
    if env_path.exists():
        prior = json.loads(env_path.read_text())
        for key in ("python", "torch", "cuda", "packages"):
            if prior[key] != environment[key]:
                raise RuntimeError(f"Resume environment changed ({key}); restore the recorded package versions before continuing.")
    else:
        _run_json(env_path, environment)
    sessions_path = root / "runtime_sessions.json"
    sessions = json.loads(sessions_path.read_text()) if sessions_path.exists() else []
    sessions.append({**environment, "started_utc": datetime.now(timezone.utc).isoformat()})
    _run_json(sessions_path, sessions)
    print(f"GPU: {environment['gpu']}; scheduled fits: {expected_fits(config)}", flush=True)
    print("Algebra checks:", protocol_self_test(), flush=True)
    # Reuse only immutable public source archives, never prior models/deployments.
    source_archive_cache = config.get("source_archive_cache")
    if source_archive_cache:
        for name, spec in LFWA_FILES.items():
            old_archive = Path(source_archive_cache) / name
            new_archive = root / "cache" / "source" / name
            if old_archive.is_file() and not new_archive.exists():
                if _lfwa_sha(old_archive) != spec["sha256"]:
                    raise RuntimeError(f"Public LFWA source archive hash mismatch: {name}")
                new_archive.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(old_archive, new_archive)
    source = prepare_lfwa_source(root / "cache")
    _run_json(root / "source_manifest.json", source["source_manifest"])
    records_dir = root / "records"
    records_dir.mkdir(exist_ok=True)
    local_dir = root / "local_models"
    local_dir.mkdir(exist_ok=True)
    start = time.perf_counter()
    deployment_identity = None
    auditor_identity = None

    def progress(status, current=None, error=None):
        counts = _run_completed_counts(root)
        payload = {"status": status, **counts, "scheduled_fits": expected_fits(config),
                   "scheduled_result_rows": expected_result_rows(config),
                   "updated_utc": datetime.now(timezone.utc).isoformat(), "current": current,
                   "error": error, "session_elapsed_seconds": time.perf_counter()-start}
        _run_json(root / "progress.json", payload)
        return counts["completed_fits"]

    def read_record(run_id, directory=None):
        path = (records_dir if directory is None else directory) / (run_id + ".json")
        if not path.exists():
            return None
        row = json.loads(path.read_text())
        if row.get("signature") != signature or row.get("source_hash") != source["source_hash"]:
            raise RuntimeError(f"Checkpoint identity mismatch: {run_id}")
        if row.get("deployment_artifact") != deployment_identity or row.get("auditor_sha256") != auditor_identity:
            raise RuntimeError(f"Checkpoint projection/deployment/auditor mismatch: {run_id}")
        return row

    try:
        # Cache only public pretrained weights in the new folder; old files stay read-only.
        old_torch = Path(config["auditor_cache"])
        new_torch = root / "cache" / "torch_home"
        if old_torch.is_dir():
            for src in old_torch.rglob("*.pt"):
                dst = new_torch / src.relative_to(old_torch)
                if not dst.exists():
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copyfile(src, dst)
        auditor = FaceAuditor(device=device, torchcache=new_torch)
        auditor_identity = auditor.metadata["state_sha256"]
        auditor_path = root / "auditor.json"
        if auditor_path.exists() and json.loads(auditor_path.read_text())["state_sha256"] != auditor_identity:
            raise RuntimeError("Frozen auditor weights changed; refusing to mix evaluations.")
        _run_json(root / "auditor.json", auditor.metadata)
        for seed in config["seeds"]:
            seed_root = root / "deployments" / f"seed_{seed}"
            progress("preparing_deployment", f"seed={seed}")
            deployment = prepare_lfwa_deployment(source, seed, seed_root, h=config["h"],
                                            max_participants=max(config["participant_counts"]), device=device)
            deployment_identity = {key: deployment["manifest"][key]
                                   for key in ("config_hash", "projection_sha256", "data_sha256")}
            participants = deployment["participants"]
            bank = generate_bank(seed, len(participants), config["h"], config["r"])
            clean_anchors = noisy_anchor_bank(bank, seed, 0, 0, 0)
            maps_clean = _run_clean_maps(clean_anchors, device)
            q = {split: [part[split + "_q"].numpy() for part in participants] for split in SPLITS}
            y = {split: [part[split + "_y"].numpy() for part in participants] for split in SPLITS}
            uploads = {split: [_run_matmul(block, bank["O"][i], bank["psi"][i], device)
                               for i, block in enumerate(q[split])] for split in SPLITS}
            common = {split: [_run_matmul(block, bank["O"][0], bank["psi"][0], device)
                              for block in q[split]] for split in SPLITS}
            target = config["target_participant"]
            query_upload = _run_matmul(deployment["query_q"].numpy(), bank["O"][target], bank["psi"][target], device)
            gallery = auditor.encode_gallery(deployment["gallery_images"])
            def privacy(qhat):
                result = auditor.linkage(qhat, deployment["F"], gallery,
                                         deployment["lineup_indices"], deployment["correct_positions"])
                truth = deployment["query_q"].numpy()
                result["projected_recovery_relative_error"] = float(np.linalg.norm(qhat-truth) / np.linalg.norm(truth))
                return result
            def three_attacks(B, maps):
                scores = {}
                for attack in ("MP", "AM", "OP"):
                    qhat = attack_reconstruct(bank["A"], B[target], query_upload, attack,
                        device=device, maps=maps, target=target, colluder_rotation=bank["O"][0])
                    scores[attack] = privacy(qhat)
                return {**scores["OP"], "attacks": scores}
            zero_privacy = three_attacks(clean_anchors, maps_clean)
            training_seed = _model_seed(seed, "utility-training")
            mlp_cfg = {**config["mlp"], "device": device}

            # Local models see only their own unperturbed public-SVD records.
            # Fit each once, then pool its held-out predictions for every prefix p.
            local_rows = []
            for participant in range(len(participants)):
                local_id = f"s{seed}_i{participant}_Local-only"
                local_row = read_record(local_id, local_dir)
                if local_row is None:
                    progress("training_local", local_id)
                    beginning = time.perf_counter()
                    metrics = fit_mlp(q["train"][participant], y["train"][participant],
                                      q["val"][participant], y["val"][participant],
                                      q["test"][participant], y["test"][participant],
                                      training_seed, mlp_cfg)
                    local_row = {
                        "run_id": local_id, "signature": signature,
                        "source_hash": source["source_hash"],
                        "deployment_artifact": deployment_identity, "auditor_sha256": auditor_identity,
                        "deployment_hash": deployment["manifest"]["config_hash"],
                        "outer_seed": seed, "participant": participant, "method": "Local-only",
                        "fit_count": 1, "utility": metrics,
                        "confusion_counts": _run_local_counts(metrics, y["test"][participant]),
                        "fit_seconds": time.perf_counter() - beginning,
                        "training_seed": training_seed, "privacy": None, "linkage_rate": None,
                        "finished_utc": datetime.now(timezone.utc).isoformat(),
                    }
                    _run_json(local_dir / (local_id + ".json"), local_row)
                    count = progress("training_local", local_id)
                    print(f"[{count}/{expected_fits(config)}] {local_id}: "
                          f"BA={100*metrics['balanced_accuracy']:.2f}%, "
                          f"fit={local_row['fit_seconds']:.1f}s", flush=True)
                if local_row.get("participant") != participant or local_row.get("outer_seed") != seed:
                    raise RuntimeError(f"Local checkpoint participant/seed mismatch: {local_id}")
                local_rows.append(local_row)

            for p in config["participant_counts"]:
                aggregate_id = f"s{seed}_p{p}_Local-only_l0_r0"
                if read_record(aggregate_id) is not None:
                    continue
                selected_local = local_rows[:p]
                metrics = _run_pool_local(selected_local)
                if metrics["n_records"] != sum(len(labels) for labels in y["test"][:p]):
                    raise RuntimeError("Local aggregate does not match the collaboration test cohort.")
                row = {
                    "run_id": aggregate_id, "signature": signature, "source_hash": source["source_hash"],
                    "deployment_hash": deployment["manifest"]["config_hash"],
                    "deployment_artifact": deployment_identity, "auditor_sha256": auditor_identity,
                    "outer_seed": seed, "participant_count": p, "method": "Local-only",
                    "noise_scale": 0.0, "level_index": 0, "realization": 0,
                    "balanced_accuracy": metrics["balanced_accuracy"], "accuracy": metrics["accuracy"],
                    "utility": metrics, "linkage_rate": None, "privacy": None,
                    "fit_count": 0, "aggregate_model_count": p,
                    "constituent_local_run_ids": [item["run_id"] for item in selected_local],
                    "fit_seconds": sum(item["fit_seconds"] for item in selected_local),
                    "alignment_seconds": 0.0, "gpm_converged": None, "gpm_iterations": None,
                    "gpm_last_step": None, "alignment": None,
                    "finished_utc": datetime.now(timezone.utc).isoformat(),
                }
                _run_json(records_dir / (aggregate_id + ".json"), row)
                progress("aggregating_local", aggregate_id)
            summarize_results(root, config)

            def fit(method, p, level_index, repetition, scale, inputs, private, diag=None):
                run_id = f"s{seed}_p{p}_{method}_l{level_index}_r{repetition}"
                cached = read_record(run_id)
                if cached is not None:
                    return cached
                progress("running", run_id)
                beginning = time.perf_counter()
                labels = {split: np.concatenate(y[split][:p]) for split in SPLITS}
                metrics = fit_mlp(inputs["train"], labels["train"], inputs["val"], labels["val"],
                                  inputs["test"], labels["test"], training_seed, mlp_cfg)
                row = {"run_id": run_id, "signature": signature, "source_hash": source["source_hash"],
                       "deployment_hash": deployment["manifest"]["config_hash"],
                       "deployment_artifact": deployment_identity, "auditor_sha256": auditor_identity,
                       "outer_seed": seed, "participant_count": p, "method": method,
                       "fit_count": 1,
                       "noise_scale": float(scale), "level_index": level_index, "realization": repetition,
                       "balanced_accuracy": metrics["balanced_accuracy"], "accuracy": metrics["accuracy"],
                       "linkage_rate": private["linkage_rate"] if private else None,
                       "privacy": private, "utility": metrics,
                       "fit_seconds": time.perf_counter()-beginning,
                       "alignment_seconds": diag["elapsed_seconds"] if diag else 0.0,
                       "gpm_converged": diag["converged"] if diag else None,
                       "gpm_iterations": diag["iterations"] if diag else None,
                       "gpm_last_step": diag["last_step"] if diag else None, "alignment": diag,
                       "finished_utc": datetime.now(timezone.utc).isoformat()}
                _run_json(records_dir / (run_id + ".json"), row)
                count = progress("running", run_id)
                print(f"[{count}/{expected_fits(config)}] {run_id}: BA={100*row['balanced_accuracy']:.2f}%"
                      + (f", linkage={100*row['linkage_rate']:.2f}%" if private else "")
                      + f", fit={row['fit_seconds']:.1f}s", flush=True)
                if count % 5 == 0:
                    summarize_results(root, config)
                return row

            # Independent utility fits, identical training seed; no duplicated pseudo-repetitions.
            equivalence_errors = {}
            for p in config["participant_counts"]:
                for method, blocks, maps, leakage in (("C-GDP", common, None, zero_privacy),
                                                     ("I-GDP", uploads, None, None),
                                                     ("AA-GDP", uploads, maps_clean, zero_privacy)):
                    run_id = f"s{seed}_p{p}_{method}_l0_r0"
                    if read_record(run_id) is None:
                        inputs = _run_collaboration(blocks, maps, p, device)
                        if method == "AA-GDP":
                            baseline = np.concatenate(common["test"][:p]).astype(np.float32)
                            error = float(np.linalg.norm(inputs["test"]-baseline) / np.linalg.norm(baseline))
                            if error > 1e-6:
                                raise RuntimeError(f"Noiseless AA/C equivalence failed: {error}")
                            equivalence_errors[str(p)] = error
                        fit(method, p, 0, 0, 0, inputs, leakage)
                        del inputs
            if equivalence_errors:
                _run_json(seed_root / "noiseless_equivalence.json", equivalence_errors)

            for level, v in enumerate(config["anchor_scales"], 1):
                for repetition in range(config["realizations"]):
                    pending = [p for p in config["participant_counts"]
                               if read_record(f"s{seed}_p{p}_NAA-GDP_l{level}_r{repetition}") is None]
                    if not pending:
                        continue
                    B = noisy_anchor_bank(bank, seed, level, repetition, v)
                    # p does not enter this attack or its gallery: one evaluation is reused exactly.
                    leaked = None
                    for p in pending:
                        progress("aligning", f"seed={seed}, p={p}, v={v}, realization={repetition}")
                        maps, diag = align_anchors(B[:p], device=device, tol=config["gpm_tolerance"],
                                                   max_iters=config["gpm_max_iters"])
                        inputs = _run_collaboration(uploads, maps, p, device)
                        leaked = three_attacks(B, maps)
                        fit("NAA-GDP", p, level, repetition, v, inputs, leaked, diag)
                        del inputs, maps
                    del B
            # The private-noise comparator is a separate, explicitly budgeted p=5 sweep.
            p = 5
            for level, sigma in enumerate(config["private_sigmas"], 1):
                for repetition in range(config["realizations"]):
                    if read_record(f"s{seed}_p{p}_C-GDP-private-noise_l{level}_r{repetition}") is not None:
                        continue
                    noisy = {split: [block + sigma*np.random.default_rng(_model_seed(
                        seed, "private-noise", level, repetition, split, i)).standard_normal(block.shape)
                                     for i, block in enumerate(common[split][:p])] for split in SPLITS}
                    # Select the actual uploaded record, including that record's exact private noise.
                    query_y = np.stack([noisy[location["split"]][target][location["position"]]
                                        for location in deployment["query_locations"]])
                    qhat = _run_matmul(query_y-bank["psi"][0], bank["O"][0].T, np.zeros(config["h"]), device)
                    leaked = privacy(qhat)
                    leaked["attacks"] = {"known-common-transform": dict(leaked)}
                    inputs = _run_collaboration(noisy, None, p, device)
                    fit("C-GDP-private-noise", p, level, repetition, sigma, inputs, leaked)
                    del noisy, inputs
            summarize_results(root, config)
            del deployment, participants, q, y, uploads, common, bank, clean_anchors, maps_clean, local_rows
            gc.collect()
            torch.cuda.empty_cache()
        raw = summarize_results(root, config)
        assert len(raw) == expected_result_rows(config)
        counts = _run_completed_counts(root)
        assert counts["completed_fits"] == expected_fits(config)
        assert counts["completed_local_fits"] == len(config["seeds"]) * max(config["participant_counts"])
        assert raw[raw.method == "NAA-GDP"].groupby("participant_count").size().eq(len(config["seeds"]) * len(config["anchor_scales"]) * config["realizations"]).all()
        progress("complete")
        print(f"Completed all {counts['completed_fits']} fitted models and {len(raw)} main result rows. "
              f"Latest results: {root}", flush=True)
        return raw
    except BaseException as exc:
        progress("interrupted" if isinstance(exc, KeyboardInterrupt) else "failed", error=str(exc))
        summarize_results(root, config)
        raise


## Numerical checks and execution
Run all after selecting A100 GPU. Full-sweep results stay separate from CelebA and the earlier LFWA experiment.


In [ ]:
print(protocol_self_test())
print(attack_self_test())
assert expected_fits(CONFIG) == 440
assert expected_result_rows(CONFIG) == 420
results = run_experiment(CONFIG, CODE_SHA256)


In [ ]:
display(pd.read_csv(Path(CONFIG["output_root"])/"summary.csv"))
display(pd.read_csv(Path(CONFIG["output_root"])/"attack_summary.csv"))
